In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

Mounted at /content/drive
  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-7q1oy880
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-7q1oy880
  Resolved https://github.com/facebookresearch/detectron2.git to commit 8a9d885b3d4dcf1bef015f0593b872ed8d32b4ab
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 30.3 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x

<h2> Import Libaraies

In [ ]:
!pip install easyocr img2table reportlab python-docx opencv-python-headless -q
!pip install paddlepaddle -q
!pip install paddleocr -q
!pip install easyocr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65

In [ ]:
# ===== Core =====
import os
import json
import glob
import shutil
import random
from pathlib import Path

# ===== Numeric & Image =====
import numpy as np
import cv2
from PIL import Image

# ===== Torch =====
import torch

# ===== Detectron2 =====
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, HookBase, DefaultPredictor
from detectron2 import model_zoo

from detectron2.data import (
    DatasetMapper,
    build_detection_train_loader,
    build_detection_test_loader,
    MetadataCatalog
)
from detectron2.data.datasets import register_coco_instances
import detectron2.data.transforms as T

from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.utils.visualizer import Visualizer, ColorMode

# ===== OCR =====
import easyocr

# ===== Table extraction =====
from img2table.document import Image as Img2TableImage
from img2table.ocr import EasyOCR as Img2TableEasyOCR

# ===== PDF =====
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Table as RLTable,
    TableStyle,
    Paragraph,
    Spacer
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm

# ===== Utils =====
from tqdm import tqdm

In [ ]:
print("Detectron2 version:", detectron2.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Detectron2 version: 0.6
PyTorch version: 2.10.0+cu128
CUDA available: True


<H2> Convert + Split train/val

In [ ]:
def convert_and_split(image_dir, output_dir, val_ratio=0.15, seed=42):
    categories = [
        {"id": 1, "name": "PartDrawing"},
        {"id": 2, "name": "Note"},
        {"id": 3, "name": "Table"},
    ]
    cat_map = {"PartDrawing": 1, "Note": 2, "Table": 3}

    all_data = []
    for lm_file in sorted(glob.glob(f"{image_dir}/*.json")):
        with open(lm_file) as f:
            data = json.load(f)

        if "shapes" not in data:
            continue
        all_data.append((lm_file, data))

    random.seed(seed)
    random.shuffle(all_data)
    n_val = max(1, int(len(all_data) * val_ratio))
    splits = {"val": all_data[:n_val], "train": all_data[n_val:]}

    os.makedirs(output_dir, exist_ok=True)

    for split, items in splits.items():
        images, annotations, ann_id = [], [], 1
        for img_id, (lm_file, data) in enumerate(items, 1):
            img_path = os.path.join(image_dir, data["imagePath"])
            if not os.path.exists(img_path):
                print(f"WARNING: image not found: {img_path}")
                continue
            w, h = Image.open(img_path).size
            images.append({
                "id": img_id,
                "file_name": data["imagePath"],
                "width": w, "height": h
            })
            for shape in data["shapes"]:
                if shape["label"] not in cat_map:
                    print(f"WARNING: unknown label '{shape['label']}' — bỏ qua")
                    continue
                pts = shape["points"]
                xs = [p[0] for p in pts]
                ys = [p[1] for p in pts]
                x1, y1 = min(xs), min(ys)
                bw = max(xs) - x1
                bh = max(ys) - y1
                annotations.append({
                    "id": ann_id, "image_id": img_id,
                    "category_id": cat_map[shape["label"]],
                    "bbox": [x1, y1, bw, bh],
                    "area": bw * bh, "iscrowd": 0
                })
                ann_id += 1

        out_path = os.path.join(output_dir, f"annotations_{split}.json")
        with open(out_path, "w") as f:
            json.dump({"images": images, "annotations": annotations,
                       "categories": categories}, f, indent=2)
        print(f"{split}: {len(images)} ảnh, {len(annotations)} bbox → {out_path}")

BASE = "/content/drive/MyDrive/engineering_drawing"
convert_and_split(
    image_dir=f"{BASE}/images",
    output_dir=f"{BASE}/annotations"
)


val: 8 ảnh, 49 bbox → /content/drive/MyDrive/engineering_drawing/annotations/annotations_val.json
train: 50 ảnh, 435 bbox → /content/drive/MyDrive/engineering_drawing/annotations/annotations_train.json


<h2> Register dataset COCO

In [ ]:
BASE = "/content/drive/MyDrive/engineering_drawing"

register_coco_instances(
    "drawing_train", {},
    os.path.join(BASE, "annotations", "annotations_train.json"),
    os.path.join(BASE, "images")
)
register_coco_instances(
    "drawing_val", {},
    os.path.join(BASE, "annotations", "annotations_val.json"),
    os.path.join(BASE, "images")
)

<h2>Train Model

In [ ]:
class TqdmHook(HookBase):
    def before_train(self):
        self.pbar = tqdm(total=self.trainer.max_iter,
                         initial=self.trainer.start_iter,
                         desc="Training", unit="iter", dynamic_ncols=True)
    def after_step(self):
        try:
            loss = self.trainer.storage.history("total_loss").latest()
        except:
            loss = 0.0
        self.pbar.update(1)
        self.pbar.set_postfix(loss=f"{loss:.4f}", refresh=True)
    def after_train(self):
        if hasattr(self, "pbar"):
            self.pbar.close()

class AugmentedTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(cfg, is_train=True, augmentations=[
            T.RandomFlip(horizontal=True, vertical=False),
            T.RandomFlip(horizontal=False, vertical=True),
            T.RandomRotation(angle=[-10, 10]),
            T.RandomBrightness(0.7, 1.3),
            T.RandomContrast(0.7, 1.3),
            T.RandomSaturation(0.8, 1.2),
            T.RandomLighting(0.5),
            T.ResizeShortestEdge(
                [640, 672, 704, 736, 768, 800],
                max_size=1333, sample_style="choice"),
        ])
        return build_detection_train_loader(cfg, mapper=mapper)

    def build_hooks(self):
        hooks = super().build_hooks()
        hooks.insert(-1, TqdmHook())
        return hooks

from detectron2.data import DatasetCatalog, MetadataCatalog
for ds in ["drawing_train", "drawing_val"]:
    if ds in DatasetCatalog:
        DatasetCatalog.remove(ds)
        MetadataCatalog.remove(ds)

register_coco_instances("drawing_train", {},
    f"{BASE}/annotations/annotations_train.json", f"{BASE}/images")
register_coco_instances("drawing_val", {},
    f"{BASE}/annotations/annotations_val.json",   f"{BASE}/images")


cfg = get_cfg()

cfg.merge_from_file(model_zoo.get_config_file(
    "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("drawing_train",)
cfg.DATASETS.TEST  = ("drawing_val",)
cfg.DATALOADER.NUM_WORKERS = 2

# ── Chỉ detect bbox, không cần mask ───────────────────────
cfg.MODEL.MASK_ON = False

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3

# ── Anchor: giữ default, thêm ratio 3.0 cho Note dạng ngang
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.25, 0.5, 1.0, 2.0, 4.0]]

# ── RepeatFactor để oversample Note ───────────────────────
cfg.DATALOADER.SAMPLER_TRAIN   = "RepeatFactorTrainingSampler"
cfg.DATALOADER.REPEAT_THRESHOLD = 0.001

# ── Solver ────────────────────────────────────────────────
cfg.SOLVER.IMS_PER_BATCH     = 2
cfg.SOLVER.BASE_LR           = 0.00025
cfg.SOLVER.MAX_ITER          = 8000
cfg.SOLVER.STEPS             = (5600, 7200)
cfg.SOLVER.GAMMA             = 0.1
cfg.SOLVER.CHECKPOINT_PERIOD = 500
cfg.SOLVER.WARMUP_ITERS      = 800
cfg.SOLVER.WARMUP_METHOD     = "linear"

# ── Model ─────────────────────────────────────────────────
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST    = 0.5
cfg.MODEL.ROI_HEADS.POSITIVE_FRACTION    = 0.33

# ── RPN: tăng proposals để không miss Note nhỏ ────────────
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN  = 4000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 2000
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST   = 2000
cfg.MODEL.RPN.POST_NMS_TOPK_TEST  = 1000

cfg.TEST.EVAL_PERIOD = 500

cfg.OUTPUT_DIR = f"{BASE}/weights_v7"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("=" * 55)
print("Model  : Cascade R-CNN + ResNet-50 + FPN")
print("Sampler: RepeatFactorTrainingSampler (boost Note)")
print("Anchor : [0.25, 0.5, 1.0, 2.0, 4.0]")
print(f"Iter   : {cfg.SOLVER.MAX_ITER} | LR: {cfg.SOLVER.BASE_LR}")
print(f"Warmup : {cfg.SOLVER.WARMUP_ITERS} | Decay: {cfg.SOLVER.STEPS}")
print(f"Output : {cfg.OUTPUT_DIR}")
print("=" * 55 + "\n")

trainer = AugmentedTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

Model  : Cascade R-CNN + ResNet-50 + FPN
Sampler: RepeatFactorTrainingSampler (boost Note)
Anchor : [0.25, 0.5, 1.0, 2.0, 4.0]
Iter   : 8000 | LR: 0.00025
Warmup : 800 | Decay: (5600, 7200)
Output : /content/drive/MyDrive/engineering_drawing/weights_v7

[04/01 14:27:30 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMa

model_final_480dd8.pkl: 288MB [00:02, 109MB/s]                           
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_predictor.0.cls_score.{bias, weight}
roi_heads.box_predictor.1.cls_score.{bias, weight}
roi_heads.box_predictor.2.cls_score.{bias, weight}
  roi_heads.mask_head.mask_fcn1.{bias, weight}
  roi_heads.mask_head.mask_fcn2.{bias, weight}
  roi_heads.mask_head.mask_fcn3.{bias, weight}
  roi_heads.mask_head.mask_fcn4.{bias, weight}
  roi_heads.mask_head.deconv.{bias, weight}
  roi_heads.mask_head.predictor.{bias, weight}


[04/01 14:27:34 d2.engine.train_loop]: Starting training from iteration 0


Training:   0%|          | 0/8000 [00:00<?, ?iter/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0401 14:27:40.435000 760 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Training:   0%|          | 20/8000 [00:18<1:16:23,  1.74iter/s, loss=4.5765]

[04/01 14:27:52 d2.utils.events]:  eta: 1:12:46  iter: 19  total_loss: 4.812  loss_cls_stage0: 1.178  loss_box_reg_stage0: 0.03041  loss_cls_stage1: 1.442  loss_box_reg_stage1: 0.06246  loss_cls_stage2: 1.174  loss_box_reg_stage2: 0.0593  loss_rpn_cls: 0.7285  loss_rpn_loc: 0.1097    time: 0.5953  last_time: 0.6175  data_time: 0.1133  last_data_time: 0.0088   lr: 6.1816e-06  max_mem: 2589M


...

Training: 100%|█████████▉| 7980/8000 [1:33:16<00:14,  1.36iter/s, loss=0.9325]

[04/01 16:00:50 d2.utils.events]:  eta: 0:00:13  iter: 7979  total_loss: 1.004  loss_cls_stage0: 0.06914  loss_box_reg_stage0: 0.1392  loss_cls_stage1: 0.03781  loss_box_reg_stage1: 0.269  loss_cls_stage2: 0.03356  loss_box_reg_stage2: 0.408  loss_rpn_cls: 0.00603  loss_rpn_loc: 0.02054    time: 0.6921  last_time: 0.7587  data_time: 0.0128  last_data_time: 0.0111   lr: 2.5e-06  max_mem: 2807M


Training: 100%|██████████| 8000/8000 [1:33:36<00:00,  2.21s/iter, loss=0.7773]

[04/01 16:01:10 d2.utils.events]:  eta: 0:00:00  iter: 7999  total_loss: 0.9073  loss_cls_stage0: 0.0574  loss_box_reg_stage0: 0.126  loss_cls_stage1: 0.03947  loss_box_reg_stage1: 0.2488  loss_cls_stage2: 0.0276  loss_box_reg_stage2: 0.3743  loss_rpn_cls: 0.004564  loss_rpn_loc: 0.0236    time: 0.6922  last_time: 0.7036  data_time: 0.0133  last_data_time: 0.0262   lr: 2.5e-06  max_mem: 2807M
[04/01 16:01:10 d2.engine.hooks]: Overall training speed: 7998 iterations in 1:32:16 (0.6922 s / it)
[04/01 16:01:10 d2.engine.hooks]: Total training time: 1:33:28 (0:01:12 on hooks)
[04/01 16:01:10 d2.data.datasets.coco]: Loaded 8 images in COCO format from /content/drive/MyDrive/engineering_drawing/annotations/annotations_val.json
[04/01 16:01:10 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[04/01 16:01:10 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.co

Training: 100%|██████████| 8000/8000 [1:33:36<00:00,  1.42iter/s, loss=0.7773]


<h2> Evaluate mAP

In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor = DefaultPredictor(cfg)
evaluator = COCOEvaluator("drawing_val", output_dir=cfg.OUTPUT_DIR)
loader    = build_detection_test_loader(cfg, "drawing_val")
print(inference_on_dataset(predictor.model, loader, evaluator))

[04/01 16:18:19 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /content/drive/MyDrive/engineering_drawing/weights_v7/model_final.pth ...
[04/01 16:18:21 d2.data.datasets.coco]: Loaded 8 images in COCO format from /content/drive/MyDrive/engineering_drawing/annotations/annotations_val.json
[04/01 16:18:21 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[04/01 16:18:21 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[04/01 16:18:21 d2.data.common]: Serializing 8 elements to byte tensors and concatenating them all ...
[04/01 16:18:21 d2.data.common]: Serialized dataset takes 0.00 MiB
[04/01 16:18:21 d2.evaluation.evaluator]: Start inference on 8 batches
[04/01 16:18:22 d2.evaluation.evaluator]: Total inference time: 0:00:00.478097 (0.159366 s / iter per device, on 1 devices)
[04/01 16:18:22 d2.eva

<h2> Hugging Face

In [ ]:
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import HfApi
import os

BASE  = "/content/drive/MyDrive/........"
TOKEN = "........."
REPO  = "........"

api = HfApi()

# Tạo repo nếu chưa có
api.create_repo(repo_id=REPO, repo_type="model",
                token=TOKEN, exist_ok=True)

# Upload tronjng số lên
api.upload_file(
    path_or_fileobj=f"{BASE}/weights_v7/model_final.pth",
    path_in_repo="model_final.pth",
    repo_id=REPO, repo_type="model", token=TOKEN,
)

api.upload_file(
    path_or_fileobj=f"{BASE}/annotations/annotations_train.json",
    path_in_repo="annotations_train.json",
    repo_id=REPO, repo_type="model", token=TOKEN,
)

print(f"✅ Upload xong! Model tại: https://huggingface.co/{REPO}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eights_v7/model_final.pth:   0%|          |  835kB /  553MB            

✅ Upload xong! Model tại: https://huggingface.co/Asuranosuke/Detect-Info


In [ ]:
#Kiểm tra download được không
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id=REPO,
    filename="model_final.pth",
    token=TOKEN
)
print(f"✅ Download OK: {path}")
import os
print(f"   Size: {os.path.getsize(path)/1e6:.1f} MB")

model_final.pth:   0%|          | 0.00/553M [00:00<?, ?B/s]

✅ Download OK: /root/.cache/huggingface/hub/models--Asuranosuke--Detect-Info/snapshots/ea09102d20c4c2a5389e8238ced54799599c2f86/model_final.pth
   Size: 552.5 MB
